# Healthcare Claims Agentic Loop

A guided walkthrough of the synthetic claims dataset, provider edit analysis, policy guides, and the Claude tool-use loop.

> All records and policy files are fictional training examples, not clinical, coverage, or adjudication guidance.

Run the cells from top to bottom. The first cells confirm which Python kernel is active and that the notebook can find the project files.

In [7]:
import sys
print(sys.executable)

/Users/rajanidevineni/ClaudeCertification/projects/customer-support-agent/.venv/bin/python


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "healthcare_agent").is_dir() and (ROOT.parent / "healthcare_agent").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "healthcare_agent").is_dir(), "Set the notebook working directory to the project root or healthcare_agent folder."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

Project root: /Users/rajanidevineni/ClaudeCertification/projects/customer-support-agent


## 1. Load the Synthetic Data

Claims reference providers through `provider_id`. Each claim stores its fired edits, including the `rule_key` used to find a workflow guide.

In [ ]:
import json

claims_data = json.loads((ROOT / "healthcare_agent/claims.json").read_text(encoding="utf-8"))
providers_data = json.loads((ROOT / "healthcare_agent/providers.json").read_text(encoding="utf-8"))
claims = claims_data["claims"]
providers = providers_data["providers"]

assert len(claims) == 30
assert all(claim["edits_fired"] for claim in claims)
assert {claim["provider_id"] for claim in claims} <= {provider["provider_id"] for provider in providers}
print(f"Loaded {len(claims)} claims and {len(providers)} providers.")

Loaded 30 claims and 6 providers.


## 2. Inspect a Claim and Its Fired Edit

Each claim can have one or more fired edits. Every edit has a `rule_key` that connects it to a Markdown guide in `healthcare_agent/edit_rules/`.

In [11]:
claim = claims[0]
edit = claim["edits_fired"][0]
claim["claim_id"], claim["provider_id"], edit

('CLM-1001',
 'PRV-201',
 {'edit_id': 'CLM-1001-E01',
  'edit_code': 'EDIT-PA-01',
  'policy_id': 'CP-101',
  'rule_key': 'authorization',
  'category': 'prior_authorization',
  'disposition': 'review_required',
  'note': 'Authorization reference is missing from mock intake.'})

## 3. Read the Matching Rule Guide

The `rule_key` is a lookup key, not a policy decision. Here we use it to open the corresponding local Markdown file.

In [ ]:
edit = claims[0]["edits_fired"][0]
rule_path = ROOT / "healthcare_agent" / "edit_rules" / f"{edit['rule_key']}.md"
print(f"Rule file: {rule_path.name}\n")
print(rule_path.read_text(encoding="utf-8"))

Rule file: authorization.md

# Authorization Edit

Synthetic workflow guidance only; this is not a real coverage rule.

- Check whether an authorization reference is present and matches the service record.
- If missing, request the authorization record or route the edit for human review.
- Do not infer approval, coverage, or medical necessity from this edit alone.



## 4. Count Edits for One Provider

This step is ordinary Python, not an LLM call. It makes the aggregation transparent: filter claims by provider, flatten their fired edits, then count by `rule_key`.

In [13]:
from collections import Counter

provider_id = "PRV-201"
provider_claims = [claim for claim in claims if claim["provider_id"] == provider_id]
provider_edits = [
    (claim, edit)
    for claim in provider_claims
    for edit in claim["edits_fired"]
]
rule_counts = Counter(edit["rule_key"] for _, edit in provider_edits)
category_counts = Counter(edit["category"] for _, edit in provider_edits)
top_count = max(rule_counts.values(), default=0)
top_rules = sorted(rule for rule, count in rule_counts.items() if count == top_count)

print(f"Provider {provider_id}: {len(provider_claims)} claims, {len(provider_edits)} fired edits")
print(f"Most frequent rule(s): {top_rules} ({top_count} each)\n")
for rule, count in sorted(rule_counts.items(), key=lambda item: (-item[1], item[0])):
    claim_ids = sorted({claim["claim_id"] for claim, edit in provider_edits if edit["rule_key"] == rule})
    print(f"{rule}: {count} edit(s); claims: {', '.join(claim_ids)}")

print("\nCategory counts:", dict(category_counts))

Provider PRV-201: 5 claims, 5 fired edits
Most frequent rule(s): ['utilization'] (2 each)

utilization: 2 edit(s); claims: CLM-1007, CLM-1019
authorization: 1 edit(s); claims: CLM-1001
coding_provider: 1 edit(s); claims: CLM-1013
documentation: 1 edit(s); claims: CLM-1025

Category counts: {'prior_authorization': 1, 'possible_duplicate': 2, 'provider_configuration': 1, 'medical_necessity_documentation': 1}


## 5. Compare with the Agent's Deterministic Tool

The demo exposes the same aggregation as a tool handler. Calling the Python function directly lets us compare its result with our notebook calculation without contacting Claude.

In [ ]:
from healthcare_agent import agent as demo

summary_json, is_error = demo.summarize_provider_edits(provider_id)
assert not is_error
summary = json.loads(summary_json)
assert summary["rule_counts"] == dict(rule_counts)
print(json.dumps(summary, indent=2))

{
  "provider_id": "PRV-201",
  "claim_count": 5,
  "claim_ids": [
    "CLM-1001",
    "CLM-1007",
    "CLM-1013",
    "CLM-1019",
    "CLM-1025"
  ],
  "total_fired_edits": 5,
  "rule_counts": {
    "authorization": 1,
    "coding_provider": 1,
    "documentation": 1,
    "utilization": 2
  },
  "category_counts": {
    "medical_necessity_documentation": 1,
    "possible_duplicate": 2,
    "prior_authorization": 1,
    "provider_configuration": 1
  },
  "most_frequent_rule_keys": [
    "utilization"
  ],
  "rule_keys_to_read": [
    "authorization",
    "coding_provider",
    "documentation",
    "utilization"
  ]
}


## 6. Read Every Fired Rule Guide

Use the keys returned by the summary tool to load the corresponding Markdown files. This step still calls local Python handlers directly; Claude has not been called yet.

In [17]:
for rule_key in summary["rule_keys_to_read"]:
    policy_json, is_error = demo.get_edit_policy(rule_key)
    if is_error:
        raise RuntimeError(policy_json)
    policy = json.loads(policy_json)
    print(f"\n{'=' * 12} {rule_key} {'=' * 12}")
    print(policy["policy_text"])


============ authorization ============
# Authorization Edit

Synthetic workflow guidance only; this is not a real coverage rule.

- Check whether an authorization reference is present and matches the service record.
- If missing, request the authorization record or route the edit for human review.
- Do not infer approval, coverage, or medical necessity from this edit alone.


============ coding_provider ============
# Coding and Provider Edit

Synthetic workflow guidance only; no real code-pair or credentialing rules are defined.

- Verify service and diagnosis coding with a qualified coding reviewer.
- Check provider specialty, credential, and service-location records in their source systems.
- Route unresolved configuration questions to human review; do not change claim outcomes.


============ documentation ============
# Documentation Edit

Synthetic workflow guidance only; this is not a real clinical policy.

- Identify the document type requested by the edit.
- Queue a mock do

## 7. Run the Live Agentic Loop

The next code cell asks for `ANTHROPIC_API_KEY` using hidden input if it is not already set in this kernel. It does not save the key in the notebook, but it remains in kernel memory until shutdown. The following cell makes a real Anthropic API request. The prompt is read-only: it asks for counts and rule summaries, not mock actions.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key (hidden input): ")
print("API key loaded for this kernel session.")

In [ ]:
prompt = (
    f"For provider {provider_id}, find which edit rule fires most often across the sample claims. "
    "Show counts and claim IDs, then read and summarize the guidance for every rule that fired. "
    "Do not change any claim or queue a task."
)
answer = demo.run_agent(prompt, verbose=True)
print(answer)

### Reading the trace

- `calling tool: summarize_provider_edits` means Python performed the deterministic provider aggregation.
- `calling tool: get_edit_policy` shows the model requested local rule-file contents. Independent rules may be requested in the same turn.
- `stop_reason=end_turn` means Claude has finished its final text response.

The model chooses what to request; Python handlers compute counts, validate mock actions, and read files. The final response is not itself a source of truth for the counts; compare it with the Python summary above.

## 8. Run the Offline Tests (Optional)

This cell runs the focused tests with the same Python interpreter as the notebook. The fake client makes no API calls.

In [ ]:
import subprocess

test_run = subprocess.run(
    [sys.executable, "-m", "pytest", "healthcare_agent/tests/test_agent.py", "-q"],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(test_run.stdout)
if test_run.returncode:
    print(test_run.stderr)
    raise RuntimeError("Healthcare demo tests failed.")

......                                                                   [100%]
6 passed in 0.28s

